# NEU-CLS v2 audit and training-only EDA

This notebook verifies the final reviewed `neu_cls_v2` manifest, then visualises the **training split only**. Locked test images are deliberately not inspected to choose preprocessing, augmentations, or hypotheses. The manifest was prepared with the reviewed duplicate-group map and is not regenerated here.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from industrial_surface_defects.constants import CLASS_NAMES
from industrial_surface_defects.data import validate_manifest
from industrial_surface_defects.transforms import build_transform, denormalize

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
MANIFEST = PROJECT_ROOT / "data/manifests/neu_cls_v2.csv"
DATA_ROOT = PROJECT_ROOT / "data/raw/NEU-CLS"
AUDIT = MANIFEST.with_suffix(".audit.json")
assert MANIFEST.exists(), "Final reviewed manifest is missing; follow the README runbook."
manifest = validate_manifest(MANIFEST)
active = manifest.loc[manifest["status"] == "ok"].copy()
train = active.loc[active["split"] == "train"].copy()
audit = json.loads(AUDIT.read_text())
audit

## Check 1 — corpus and split invariants

The official archive contains 1,800 readable 200×200 grayscale images. One exact duplicate is retained as an excluded provenance row, leaving 1,799 active images. All confirmed shifted-duplicate groups must remain within one active split.

In [ ]:
counts = active.groupby(["split", "class_name"]).size().unstack(fill_value=0)
counts = counts.reindex(index=["train", "val", "test"], columns=CLASS_NAMES)
display(counts)
expected_counts = pd.DataFrame(
    {name: [209 if name == "patches" else 210, 30, 60] for name in CLASS_NAMES},
    index=["train", "val", "test"],
)
pd.testing.assert_frame_equal(counts, expected_counts, check_names=False)
assert counts.sum(axis=1).to_dict() == {"train": 1259, "val": 180, "test": 360}
assert len(manifest) == 1800 and len(active) == 1799
assert audit["image_count"] == 1800 and audit["effective_image_count"] == 1799
assert audit["exact_duplicate_extra_images"] == 1
assert audit["effective_class_counts"]["patches"] == 299
assert audit["near_duplicate_threshold"] == 12
assert audit["near_duplicate_candidates"] == 13
assert audit["near_duplicate_confirmed_shifted_pairs"] == 4
assert audit["near_duplicate_unresolved_candidates"] == 0
assert active.groupby("sha256")["split"].nunique().max() == 1
assert active.groupby("group_id")["split"].nunique().max() == 1

by_path = manifest.set_index("path")
assert by_path.loc["NEU-CLS/Pa_101.bmp", "status"] == "ok"
assert by_path.loc["NEU-CLS/Pa_105.bmp", "status"] == "duplicate_quarantined"
assert by_path.loc["NEU-CLS/Pa_105.bmp", "split"] == "excluded"
assert by_path.loc["NEU-CLS/Pa_101.bmp", "sha256"] == by_path.loc["NEU-CLS/Pa_105.bmp", "sha256"]

ps_pair = by_path.loc[["NEU-CLS/PS_171.bmp", "NEU-CLS/PS_292.bmp"]]
assert ps_pair["group_id"].nunique() == 1 and ps_pair["split"].nunique() == 1
in_pair = by_path.loc[["NEU-CLS/In_188.bmp", "NEU-CLS/In_210.bmp"]]
assert in_pair["group_id"].nunique() == 2
assert in_pair["split"].nunique() == 1 and in_pair["split"].iloc[0] == "train"

In [ ]:
ax = counts.T.plot(kind="bar", figsize=(9, 4), rot=30)
ax.set(title="Fixed split composition", xlabel="Defect class", ylabel="Images")
plt.tight_layout()
plt.show()

## Check 2 — representative training images

A fixed seeded sample makes within-class variation and visually similar classes inspectable without cherry-picking.

In [ ]:
sample = (
    train.groupby("class_name", group_keys=False)
    .sample(n=5, random_state=42)
    .sort_values(["class_index", "path"])
)
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(12, 13))
for row, class_name in enumerate(CLASS_NAMES):
    rows = sample.loc[sample["class_name"] == class_name]
    for column, item in enumerate(rows.itertuples()):
        with Image.open(DATA_ROOT / item.path) as image:
            axes[row, column].imshow(image.convert("L"), cmap="gray", vmin=0, vmax=255)
        axes[row, column].set_title(Path(item.path).name, fontsize=7)
        axes[row, column].axis("off")
    axes[row, 0].text(
        -0.15, 0.5, class_name, rotation=90, va="center", transform=axes[row, 0].transAxes
    )
fig.suptitle("Seeded training examples — not test images", fontsize=14)
plt.tight_layout()
plt.show()

## Check 3 — intensity shortcuts

Large differences in global brightness or contrast can let a classifier exploit acquisition shortcuts instead of defect morphology. This is a diagnostic, not evidence of causality.

In [ ]:
def image_statistics(relative_path):
    with Image.open(DATA_ROOT / relative_path) as source:
        pixels = np.asarray(source.convert("L"), dtype=np.float32) / 255.0
    return pixels.mean(), pixels.std(), np.quantile(pixels, 0.05), np.quantile(pixels, 0.95)


stats = np.asarray([image_statistics(path) for path in train["path"]])
train_stats = train[["path", "class_name"]].reset_index(drop=True).copy()
train_stats[["mean", "std", "p05", "p95"]] = stats
display(train_stats.groupby("class_name")[["mean", "std", "p05", "p95"]].agg(["mean", "std"]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for class_name, rows in train_stats.groupby("class_name"):
    axes[0].hist(rows["mean"], bins=25, alpha=0.35, label=class_name)
    axes[1].hist(rows["std"], bins=25, alpha=0.35, label=class_name)
axes[0].set(title="Global mean intensity", xlabel="Mean in [0, 1]", ylabel="Images")
axes[1].set(title="Global contrast proxy", xlabel="Pixel standard deviation")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Check 4 — augmentation visual QA

The camera-aware pipeline is acceptable only if seeded examples preserve the semantic defect label. Review this grid as a team before the first registered run.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
camera_transform = build_transform(224, "camera_aware", train=True)
examples = train.groupby("class_name", group_keys=False).sample(n=1, random_state=42)
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(12, 13))
for row, item in enumerate(examples.sort_values("class_index").itertuples()):
    with Image.open(DATA_ROOT / item.path) as source:
        source = source.convert("L")
        axes[row, 0].imshow(source, cmap="gray", vmin=0, vmax=255)
        for column in range(1, 5):
            augmented = denormalize(camera_transform(source)).mean(0)
            axes[row, column].imshow(augmented, cmap="gray", vmin=0, vmax=1)
    for axis in axes[row]:
        axis.axis("off")
    axes[row, 0].set_title(item.class_name)
axes[0, 0].set_ylabel("original")
fig.suptitle("Original plus four camera-aware draws per training class", fontsize=14)
plt.tight_layout()
plt.show()

## Review checklist before registered training

1. The v2 data decisions are frozen: one exact duplicate is excluded, all 13 pHash candidates are resolved, and four confirmed shifted pairs share groups. Do not alter the split after observing model performance.
2. Can the class be inferred from global intensity rather than local texture? Treat this as a shortcut hypothesis; if explored, keep any histogram-only sanity baseline outside the primary factorial comparison.
3. Do augmentations erase or morph the defect? Revise and version the registered ranges **before** locked test evaluation, then disclose the deviation.
4. NEU-CLS contains only known-defect crops. One-vs-rest false positives are wrong defect-type assignments, not false alarms on normal steel; localisation is not evaluated.